In [1]:
import pandas as pd

df = pd.read_csv("../data/dataset.csv")

df.shape

(11055, 32)

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 11055 entries, 0 to 11054
Data columns (total 32 columns):
 #   Column                       Non-Null Count  Dtype
---  ------                       --------------  -----
 0   index                        11055 non-null  int64
 1   having_IPhaving_IP_Address   11055 non-null  int64
 2   URLURL_Length                11055 non-null  int64
 3   Shortining_Service           11055 non-null  int64
 4   having_At_Symbol             11055 non-null  int64
 5   double_slash_redirecting     11055 non-null  int64
 6   Prefix_Suffix                11055 non-null  int64
 7   having_Sub_Domain            11055 non-null  int64
 8   SSLfinal_State               11055 non-null  int64
 9   Domain_registeration_length  11055 non-null  int64
 10  Favicon                      11055 non-null  int64
 11  port                         11055 non-null  int64
 12  HTTPS_token                  11055 non-null  int64
 13  Request_URL                  11055 non-null  int64
 14  U

In [3]:
df.columns.tolist()

['index',
 'having_IPhaving_IP_Address',
 'URLURL_Length',
 'Shortining_Service',
 'having_At_Symbol',
 'double_slash_redirecting',
 'Prefix_Suffix',
 'having_Sub_Domain',
 'SSLfinal_State',
 'Domain_registeration_length',
 'Favicon',
 'port',
 'HTTPS_token',
 'Request_URL',
 'URL_of_Anchor',
 'Links_in_tags',
 'SFH',
 'Submitting_to_email',
 'Abnormal_URL',
 'Redirect',
 'on_mouseover',
 'RightClick',
 'popUpWidnow',
 'Iframe',
 'age_of_domain',
 'DNSRecord',
 'web_traffic',
 'Page_Rank',
 'Google_Index',
 'Links_pointing_to_page',
 'Statistical_report',
 'Result']

In [4]:
df["Result"].value_counts()

Result
 1    6157
-1    4898
Name: count, dtype: int64

## Observations sur le dataset

- Le fichier contient 11 055 lignes et 32 colonnes (30 caracteristiques + index + Result).
- Aucune valeur manquante : toutes les colonnes sont "non-null" sur les 11 055 lignes.
- Toutes les colonnes sont deja numeriques (int64).
- La colonne cible est `Result` : 1 = site legitime, -1 = site de phishing.
- Repartition des classes : 6 157 legitimes (1) vs 4 898 phishing (-1) — dataset relativement equilibre.

In [5]:
df = df.drop(columns=["index"])
df.shape   # doit maintenant afficher (11055, 31)

(11055, 31)

In [8]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["Result"])
y = df["Result"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Taille entraînement :", X_train.shape)
print("Taille test :", X_test.shape)

Taille entraînement : (8844, 30)
Taille test : (2211, 30)


In [9]:
from sklearn.linear_model import LogisticRegression

modele = LogisticRegression(max_iter=1000)
modele.fit(X_train, y_train)

print("Entraînement terminé.")

Entraînement terminé.


In [10]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = modele.predict(X_test)

print("Taux de réussite (accuracy) :", accuracy_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred))

Taux de réussite (accuracy) : 0.9289914066033469

              precision    recall  f1-score   support

          -1       0.94      0.90      0.92       980
           1       0.92      0.95      0.94      1231

    accuracy                           0.93      2211
   macro avg       0.93      0.93      0.93      2211
weighted avg       0.93      0.93      0.93      2211



In [11]:
print(confusion_matrix(y_test, y_pred))

[[ 884   96]
 [  61 1170]]


## Interpretation des resultats

- Taux de reussite global : 92.9%
- Sur 980 sites de phishing reels dans le jeu de test, 884 sont correctement 
  detectes et 96 sont manques (faux negatifs, rappel de 90%).
- Sur 1231 sites legitimes, 1170 sont correctement reconnus et 61 sont 
  signales a tort comme phishing (faux positifs).
- Dans un contexte de securite, les faux negatifs (phishing non detecte) 
  sont plus critiques que les faux positifs.

In [12]:
import joblib

joblib.dump(modele, "../data/modele_phishing.pkl")
print("Modèle sauvegardé dans data/modele_phishing.pkl")

Modèle sauvegardé dans data/modele_phishing.pkl
